In [1]:
# ============================================
# SMART RAINWATER ML MODELS – FULL PIPELINE
# ============================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    accuracy_score,
    classification_report
)

import joblib

print("Libraries imported successfully")


Libraries imported successfully


In [2]:
# ----------------------------------
# LOAD AUGMENTED DATASET
# ----------------------------------

dataset_path = f"E:\\Documents\\SEM-8\\SDP\\dataset\\final_augmented_rainwater_dataset.csv"  # <-- EDIT HERE

df = pd.read_csv(dataset_path)

print("Dataset loaded")
print("Shape:", df.shape)
df.head()


Dataset loaded
Shape: (10000, 13)


,city,district,state,rainfall_intensity_mm_hr,raindrop_size_mm,rainfall_type,roof_area_m2,ph,tds,turbidity,harvestable,harvest_quantity_liters,rain_energy_joules
0,Ahmedabad,Ahmedabad,Gujarat,29.255681,2.011779,Moderate,213.461139,5.977273,2500.0,5.201643,0,0.0,0.860936
1,Jaipur,Jaipur,Rajasthan,30.330347,1.465127,Moderate,196.292969,NaN,2500.0,3.831937,0,0.0,0.673907
2,Vijayawada,Krishna,Andhra Pradesh,55.444304,3.917487,Heavy,90.557418,5.913960,2500.0,2.874449,0,0.0,6.021316
3,Mumbai,Mumbai,Maharashtra,12.741987,1.278748,Storm,51.976206,9.500000,2500.0,2.554843,0,0.0,0.103808
4,Mumbai,Mumbai,Maharashtra,13.333151,1.448621,Storm,57.645911,9.500000,2500.0,2.938052,0,0.0,0.128763


In [3]:
# ----------------------------------
# DATA CLEANING & ENCODING
# ----------------------------------

# Drop duplicates if any
df = df.drop_duplicates()

# Encode categorical columns
label_encoders = {}

for col in ["city", "district", "state", "rainfall_type"]:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

print("Encoding completed")


Encoding completed


In [4]:
# ----------------------------------
# FEATURE SET
# ----------------------------------

feature_cols = [
    "city",
    "district",
    "state",
    "rainfall_intensity_mm_hr",
    "raindrop_size_mm",
    "roof_area_m2",
    "rainfall_type"
]

X = df[feature_cols]

# Targets
y_ph = df["ph"]
y_tds = df["tds"]
y_turbidity = df["turbidity"]

y_harvestable = df["harvestable"]
y_quantity = df["harvest_quantity_liters"]

print("Features and targets prepared")


Features and targets prepared


In [10]:
# ----------------------------------
# TRAIN TEST SPLIT
# ----------------------------------

X_train, X_test, y_ph_train, y_ph_test = train_test_split(
    X, y_ph, test_size=0.2, random_state=42
)

_, _, y_tds_train, y_tds_test = train_test_split(
    X, y_tds, test_size=0.2, random_state=42
)

_, _, y_turb_train, y_turb_test = train_test_split(
    X, y_turbidity, test_size=0.2, random_state=42
)

_, _, y_harv_train, y_harv_test = train_test_split(
    X, y_harvestable, test_size=0.2, random_state=42
)

_, _, y_qty_train, y_qty_test = train_test_split(
    X, y_quantity, test_size=0.2, random_state=42
)

print("Train-test split completed")


Train-test split completed


In [11]:
# ----------------------------------
# HANDLE MISSING VALUES (CRITICAL)
# ----------------------------------

# Check missing values in targets
print("Missing values before handling:\n")
print(df[["ph", "tds", "turbidity", "harvestable", "harvest_quantity_liters"]].isna().sum())

# Fill missing pollution values with median
df["ph"].fillna(df["ph"].median(), inplace=True)
df["tds"].fillna(df["tds"].median(), inplace=True)
df["turbidity"].fillna(df["turbidity"].median(), inplace=True)

# Harvestable: default to NOT harvestable if unknown
df["harvestable"].fillna(0, inplace=True)

# Harvest quantity: set 0 where missing
df["harvest_quantity_liters"].fillna(0, inplace=True)

print("\nMissing values after handling:\n")
print(df[["ph", "tds", "turbidity", "harvestable", "harvest_quantity_liters"]].isna().sum())


Missing values before handling:

ph                         0
tds                        0
turbidity                  0
harvestable                0
harvest_quantity_liters    0
dtype: int64

Missing values after handling:

ph                         0
tds                        0
turbidity                  0
harvestable                0
harvest_quantity_liters    0
dtype: int64


In [12]:
# ----------------------------------
# RE-CREATE FEATURES & TARGETS (FIX)
# ----------------------------------

# Features
feature_cols = [
    "city",
    "district",
    "state",
    "rainfall_intensity_mm_hr",
    "raindrop_size_mm",
    "roof_area_m2",
    "rainfall_type"
]

X = df[feature_cols]

# Targets (AFTER NaN handling)
y_ph = df["ph"]
y_tds = df["tds"]
y_turbidity = df["turbidity"]
y_harvestable = df["harvestable"]
y_quantity = df["harvest_quantity_liters"]

# Safety checks
print("NaNs check:")
print("ph:", y_ph.isna().sum())
print("tds:", y_tds.isna().sum())
print("turbidity:", y_turbidity.isna().sum())
print("harvestable:", y_harvestable.isna().sum())
print("quantity:", y_quantity.isna().sum())


NaNs check:
ph: 0
tds: 0
turbidity: 0
harvestable: 0
quantity: 0


In [13]:
# ----------------------------------
# POLLUTION MODELS
# ----------------------------------

rf_ph = RandomForestRegressor(n_estimators=200, random_state=42)
rf_tds = RandomForestRegressor(n_estimators=200, random_state=42)
rf_turb = RandomForestRegressor(n_estimators=200, random_state=42)

rf_ph.fit(X_train, y_ph_train)
rf_tds.fit(X_train, y_tds_train)
rf_turb.fit(X_train, y_turb_train)

print("Pollution models trained")


Pollution models trained


In [14]:
# ----------------------------------
# HARVESTABLE CLASSIFIER
# ----------------------------------

rf_harvest = RandomForestClassifier(n_estimators=200, random_state=42)
rf_harvest.fit(X_train, y_harv_train)

print("Harvestability model trained")


Harvestability model trained


In [15]:
# ----------------------------------
# HARVEST QUANTITY REGRESSOR
# ----------------------------------

rf_quantity = RandomForestRegressor(n_estimators=200, random_state=42)
rf_quantity.fit(X_train, y_qty_train)

print("Harvest quantity model trained")


Harvest quantity model trained


In [22]:
# ----------------------------------
# EVALUATION
# ----------------------------------

print("pH R2:", r2_score(y_ph_test, rf_ph.predict(X_test)))
print("TDS R2:", r2_score(y_tds_test, rf_tds.predict(X_test)))
print("Turbidity R2:", r2_score(y_turb_test, rf_turb.predict(X_test)))
print("\nHarvestable Accuracy:",
      accuracy_score(y_harv_test, rf_harvest.predict(X_test)))
print("\nHarvest Quantity R2:",
      r2_score(y_qty_test, rf_quantity.predict(X_test)))


pH R2: 0.9788144359850081
TDS R2: 1.0
Turbidity R2: 0.38955679884027994

Harvestable Accuracy: 0.8885

Harvest Quantity R2: 0.5753704995659357


In [21]:
# ----------------------------------
# SAMPLE PREDICTION TEST (FINAL SYSTEM LOGIC)
# ----------------------------------

sample = X_test.iloc[[0]]

# Pollution predictions
ph_pred = rf_ph.predict(sample)[0]
tds_pred = rf_tds.predict(sample)[0]
turb_pred = rf_turb.predict(sample)[0]

# Harvest decision
harv_pred = rf_harvest.predict(sample)[0]

# Harvest quantity (gated)
if harv_pred == 1:
    qty_pred = rf_quantity.predict(sample)[0]
else:
    qty_pred = 0

# Energy estimation (ALWAYS calculated)
rainfall_intensity = sample["rainfall_intensity_mm_hr"].values[0]
raindrop_size = sample["raindrop_size_mm"].values[0]

energy_joules = 0.5 * (raindrop_size / 1000) * (rainfall_intensity ** 2)

# Output
print("Predicted pH:", round(ph_pred, 2))
print("Predicted TDS:", round(tds_pred, 2))
print("Predicted Turbidity:", round(turb_pred, 2))

print("Harvestable:", "Yes" if harv_pred == 1 else "No")
print("Harvest Quantity (liters):", round(qty_pred, 2))

print("Estimated Rain Energy (Joules):", round(energy_joules, 4))


Predicted pH: 8.83
Predicted TDS: 2500.0
Predicted Turbidity: 4.43
Harvestable: No
Harvest Quantity (liters): 0
Estimated Rain Energy (Joules): 3.668


In [19]:
# ----------------------------------
# SAVE MODELS TO CUSTOM PATH
# ----------------------------------

import os
import joblib

MODEL_DIR = r"E:\Documents\SEM-8\SDP\Models"

# Create directory if it doesn't exist
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(rf_ph, os.path.join(MODEL_DIR, "ph_model.pkl"))
joblib.dump(rf_tds, os.path.join(MODEL_DIR, "tds_model.pkl"))
joblib.dump(rf_turb, os.path.join(MODEL_DIR, "turbidity_model.pkl"))
joblib.dump(rf_harvest, os.path.join(MODEL_DIR, "harvestable_model.pkl"))
joblib.dump(rf_quantity, os.path.join(MODEL_DIR, "harvest_quantity_model.pkl"))
joblib.dump(label_encoders, os.path.join(MODEL_DIR, "label_encoders.pkl"))

print("✅ All models saved successfully to:")
print(MODEL_DIR)


✅ All models saved successfully to:
E:\Documents\SEM-8\SDP\Models
